In [ ]:
from datetime import datetime
from getpass import getpass

admin_rdm_url = 'https://admin.test.rdm.example.com/'
rdm_url = 'https://test.rdm.example.com/'
idp_name_1 = 'GakuNin RDM IdP'
idp_username_1 = None
idp_password_1 = None
member_idp_username = None
member_idp_password = None
skip_130mb_upload = False
default_result_path = None
close_on_fail = False
transition_timeout = 60000
skip_failed_test = True
exclude_notebooks = []
prepare_migration = False

# 機関ストレージ設定用パラメータ
target_organization = 'GakuNin RDM IdP'

storage_provider = 's3compatsigv4'
expected_s3_region = None
# S3 接続設定
s3_endpoint_url = None
s3_access_key = None
s3_secret_key = None
s3_bucket = None

# Server Side Encryption: False = NO
s3_server_side_encryption = False

# 機関ストレージの表示名（ファイルツリーに表示される名前）
institutional_storage_name = 'S3CompatSigV4 Storage'

# プロジェクト名プレフィックス
rdm_project_prefix = 'TEST-INST-{}'.format(datetime.now().strftime('%Y%m%d-%H%M%S'))

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

In [ ]:
if storage_provider == 's3compatsigv4' and s3_endpoint_url is None:
    s3_endpoint_url = input(prompt='S3 Endpoint URL')
if s3_access_key is None:
    s3_access_key = input(prompt='S3 Access Key')
if s3_secret_key is None:
    s3_secret_key = getpass(prompt='S3 Secret Key')
if s3_bucket is None:
    s3_bucket = input(prompt='S3 Bucket Name')
(storage_provider, len(s3_access_key), len(s3_secret_key), len(s3_bucket))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# GakuNinRDM 総合テスト [機関ストレージ]

- サブシステム名: 管理者 / ストレージ
- ページ/アドオン: 機関ストレージ
- 機能分類: 機関ストレージ設定・ファイル操作確認
- シナリオ名: 機関ストレージを登録し、所属ユーザーのファイル操作を確認する
- 用意するテストデータ: GRDMと管理画面のURL、機関管理者と所属ユーザーのアカウント、ストレージの接続情報

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## GakuNin RDM管理者ページのURLを開く

管理者ページが表示されること

In [ ]:
import time

async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDM管理者画面にログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
#   async def _debug(page):                                                                                         
#       # IdPリストの周辺HTMLを取得                                                                                 
#       html = await page.evaluate('() => document.getElementById("IdPList") ? document.getElementById("IdPList").outerHTML : document.body.innerHTML.substring(0, 5000)')
#       print(html)

#   await run_pw(_debug)

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    # ログアウトボタンまたはlogoutリンクが表示されること（日本語/英語環境両対応）
    logout_btn = page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link = page.locator('//*[@href="/account/logout/"]')
    await expect(logout_btn.or_(logout_link)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## サイドメニューの「機関ストレージ」を選択する

機関ストレージの設定画面が表示されること

In [ ]:
import traceback

async def _step(page):
    # サイドメニューの「機関ストレージ」をクリック（「機関ストレージのクォータ」と区別）
    link = page.locator('//a[contains(@href, "institutional_storage") and not(contains(@href, "quota"))]').first
    await expect(link).to_be_visible(timeout=transition_timeout)
    await link.click()

    # 機関ストレージ設定画面が表示されること（ラジオボタンの存在で確認）
    await expect(page.locator('//input[@type="radio"]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 試験対象のストレージ種別を選択する

指定した種別のラジオボタンが選択されること。

In [ ]:
async def _step(page):
    storage = page.locator(f'input[type="radio"][value="{storage_provider}"]')
    await storage.check()
    await expect(storage).to_be_checked()

await run_pw(_step)

## 「Save」ボタンをクリックし、確認ダイアログで確認文字列を入力して「Change」をクリックする

機関ストレージ変更の確認ダイアログが表示される。表示された確認文字列を入力し、「Change」をクリックすると対象ストレージの接続設定画面が表示されること。

※ 確認文字列はランダムに生成されるため、画面を確認して手入力する。

In [ ]:
async def _step(page):
    storage_name = page.locator('#storage_name')
    if await storage_name.is_visible() and await storage_name.is_enabled():
        await storage_name.fill(institutional_storage_name)
    await page.locator('#institutional_storage_form button[type="submit"]').click()

    confirm_text = page.locator('#bbConfirmText')
    storage_modal = page.locator(f'#{storage_provider}_modal')
    dialog = page.locator(f'#bbConfirmText:visible, #{storage_provider}_modal:visible').first
    await expect(dialog).to_be_visible(timeout=transition_timeout)
    if await confirm_text.is_visible():
        confirmation = await page.locator('.bootbox-body strong').text_content()
        await confirm_text.fill(confirmation)
        await page.locator('.bootbox button.btn-danger').click()
    await expect(storage_modal).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ストレージの接続情報を入力する

アクセスキー、シークレットキー、バケット名、暗号化の有無を指定できること。S3互換ストレージではエンドポイントURLも指定できること。

In [ ]:
async def _step(page):
    if storage_provider == 's3compatsigv4':
        await page.locator('#s3compatsigv4_endpoint_url').fill(s3_endpoint_url)
    await page.locator(f'#{storage_provider}_access_key').fill(s3_access_key)
    await page.locator(f'#{storage_provider}_secret_key').fill(s3_secret_key)
    bucket = page.locator(f'#{storage_provider}_bucket')
    await bucket.fill(s3_bucket)
    await page.locator(f'#{storage_provider}_server_side_encryption').set_checked(s3_server_side_encryption)
    await bucket.press('End')
    await expect(page.locator(f'#{storage_provider}_connect')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「Connect」ボタンをクリックして接続テストを行う

接続テストが成功すること

In [ ]:
async def _step(page):
    # Connect ボタンをクリック
    connect_btn = page.locator(f'#{storage_provider}_connect')
    await expect(connect_btn).to_be_enabled(timeout=transition_timeout)
    await connect_btn.click()

    # Save ボタンが有効になるのを待つ（接続成功の証）
    save_btn = page.locator(f'#{storage_provider}_save')
    await expect(save_btn).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「Save」ボタンをクリックして設定を保存する

機関ストレージの設定が保存されること

In [ ]:
async def _step(page):
    # Save ボタンをクリック
    save_btn = page.locator(f'#{storage_provider}_save')
    await save_btn.click()

    # モーダルが閉じるのを待つ
    await expect(page.locator(f'#{storage_provider}_modal')).to_be_hidden(timeout=transition_timeout)

    # 少し待って設定が反映されるのを確認
    await asyncio.sleep(2)

await run_pw(_step)

## 管理者画面からログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    # ログアウト（日本語/英語環境両対応）
    logout_btn = page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link = page.locator('//*[@href="/account/logout/"]')
    if await logout_btn.count() > 0:
        await logout_btn.click()
    else:
        await logout_link.click()

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # ページの読み込みを待つ
    await page.wait_for_load_state('networkidle')

    # 同意するボタンがあれば先にクリック
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count() > 0 and await consent_button.is_visible():
        await consent_button.click()
        await page.wait_for_load_state('networkidle')

    # サインインボタン、ダッシュボード、FakeCASフォームのいずれかを待つ
    sign_in_button = page.locator('//button[@data-test-sign-in-button]')
    dashboard = page.locator('//*[@data-test-create-project-modal-button]')
    fakecas_form = page.locator('#username')

    await expect(sign_in_button.or_(dashboard).or_(fakecas_form)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, member_idp_username or idp_username_1, member_idp_password or idp_password_1, transition_timeout=transition_timeout
    )

    # ダッシュボードが表示されること（プロジェクト作成ボタンで確認）
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトを作成する

「{rdm_project_prefix}-institutional」プロジェクトが作成されること

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

rdm_project_name = f'{rdm_project_prefix}-institutional'

async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, rdm_project_name, transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()

    # プロジェクトダッシュボードのナビタブが表示されること
    await expect(page.locator('#projectNavFiles')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ファイル」タブをクリックする

ファイルタブが表示されること。NII Storageの代わりに設定した機関ストレージが表示されること。

In [ ]:
async def _step(page):
    await page.locator('#projectNavFiles a').click()

    await expect(page.locator('//*[@id = "treeGrid"]')).to_be_visible(timeout=transition_timeout)

    # 少し待ってファイルツリーが読み込まれるのを確認
    await asyncio.sleep(3)

await run_pw(_step)

## テスト用ファイルを作成する

テスト用のファイルが作成されること

In [ ]:
import os

test_file_name = 'test_institutional_storage.txt'
test_file_path = os.path.join(work_dir, test_file_name)
with open(test_file_path, 'w') as f:
    if prepare_migration:
        f.write('S3 institutional storage migration fixture')
    else:
        f.write(f'Institutional Storage Test - {storage_provider}\n')
        f.write(f'Created at: {datetime.now().isoformat()}\n')
        f.write(f'Bucket: {s3_bucket}\n')
        f.write(f'Endpoint: {s3_endpoint_url}\n')

print(f'Test file created: {test_file_path}')

## ファイルをアップロードする

ファイルがアップロードされ、ファイル一覧に表示されること

In [ ]:
async def _step(page):
    # 機関ストレージプロバイダをクリックして開く
    await page.locator(f'//*[contains(text(), "{institutional_storage_name}")]').click()
    await asyncio.sleep(2)

    # アップロードボタンが表示されるのを待つ
    upload_btn = page.locator('//i[contains(@class, "fa-upload")]/../*[text() = "アップロード"]')
    await expect(upload_btn).to_be_visible(timeout=transition_timeout)

    await scripts.grdm.upload_file(page, test_file_path)

    # アップロードしたファイルが表示されるのを確認
    await expect(page.locator(f'//*[contains(text(), "{test_file_name}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (Amazon S3の場合) ファイル一覧の上部メニューから「ダウンロード」をクリックする

ダウンロードした内容がアップロードしたファイルと一致すること。WaterButlerがHTTP 302を返し、対象リージョンのSigV4署名付きURLからダウンロードされること。

In [ ]:
import filecmp
from urllib.parse import parse_qs, urlsplit

async def _step(page):
    if expected_s3_region is None:
        return

    await grdm.get_select_file_extension_locator(page, test_file_name).click()
    button = page.locator('.fangorn-toolbar-icon').get_by_text('ダウンロード', exact=True)
    async with page.expect_download(timeout=transition_timeout) as pending_download:
        async with page.expect_response(
            lambda response: (response.status == 302
                              and '/providers/osfstorage/' in urlsplit(response.url).path),
            timeout=transition_timeout,
        ) as redirect:
            await button.click()
    location = urlsplit((await redirect.value).headers['location'])
    query = parse_qs(location.query)
    print(f'HTTP 302: {location.hostname}')
    assert location.scheme == 'https'
    allowed_hosts = {f's3.{expected_s3_region}.amazonaws.com'}
    if expected_s3_region == 'us-east-1':
        allowed_hosts.add('s3.amazonaws.com')
    assert any(location.hostname in (host, f'{s3_bucket}.{host}') for host in allowed_hosts)
    assert query['X-Amz-Algorithm'] == ['AWS4-HMAC-SHA256']
    assert query['X-Amz-Credential'][0].endswith(f'/{expected_s3_region}/s3/aws4_request')
    assert filecmp.cmp(test_file_path, await (await pending_download.value).path(), shallow=False)

await run_pw(_step)

## アップロードしたファイルをクリックして内容を確認する

ファイルの内容が表示されること

In [ ]:
import os

async def _step(page):
    if 'files' in page.url and test_file_name in await page.title():
        pass
    else:
        await grdm.get_select_file_title_locator(page, test_file_name).click(timeout=transition_timeout)
        await asyncio.sleep(3)

    # ファイル詳細ページが表示されていることを確認（ファイル名ヘッダーの拡張子）
    filename_body, filename_ext = os.path.splitext(test_file_name)
    await expect(page.locator(f'//h2[contains(text(), "{filename_body}")]//*[@id = "file-ext"]')).to_have_text(filename_ext, timeout=transition_timeout)

await run_pw(_step)

## （移行試験の準備時を除く）プロジェクトを削除する

プロジェクトが削除されること

In [ ]:
async def _step(page):
    await scripts.grdm.delete_project(page)

    # ダッシュボードに戻ったことを確認
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_be_visible(timeout=transition_timeout)

if not prepare_migration:
    await run_pw(_step)

In [ ]:
if not prepare_migration:
    from datetime import datetime
    import os
    import papermill as pm
    import traceback
    from scripts.papermillHelpers import gen_run_notebook

    def make_result_dir(base_path):
        result_dir = os.path.join(base_path, 'notebooks')
        os.makedirs(result_dir, exist_ok=True)
        return result_dir

    result_dir = make_result_dir(default_result_path)

    run_notebook = gen_run_notebook(
        result_dir,
        transition_timeout,
        dict(
            rdm_url=rdm_url,
            idp_name_1=idp_name_1,
            idp_username_1=member_idp_username or idp_username_1,
            idp_password_1=member_idp_password or idp_password_1,
        ),
        skip_failed_test,
        exclude_notebooks,
    )

    result_notebooks = []
    result_dir

## （移行試験の準備時を除く）プロジェクトダッシュボードでの「ファイル基本操作」テストの実施

テスト「テスト手順-ストレージ共通-ファイル基本操作」をプロジェクトダッシュボードで実施する。

In [ ]:
if not prepare_migration:
    result_notebooks.append(run_notebook(
        'テスト手順-ストレージ共通-ファイル基本操作.ipynb',
        dict(
            enable_52gb_file_upload=False,
            target_storage_name=institutional_storage_name,
            target_storage_id='osfstorage',
            skip_130mb_upload=skip_130mb_upload,
            target_file_view='project-dashboard',
            rdm_project_name=f'{rdm_project_prefix}-dashboard',
        ),
    ))
    result_notebooks

## （移行試験の準備時を除く）ファイルタブでの「ファイル基本操作」テストの実施

テスト「テスト手順-ストレージ共通-ファイル基本操作」をファイルタブで実施する。

In [ ]:
if not prepare_migration:
    result_notebooks.append(run_notebook(
        'テスト手順-ストレージ共通-ファイル基本操作.ipynb',
        dict(
            enable_52gb_file_upload=False,
            target_storage_name=institutional_storage_name,
            target_storage_id='osfstorage',
            skip_130mb_upload=skip_130mb_upload,
            target_file_view='file-tab',
            rdm_project_name=f'{rdm_project_prefix}-filetab',
        ),
        '-file-tab',
    ))
    result_notebooks

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}